In [ ]:
'''
MINI-PROJECT 1: Data Pre-Processing and Processing
By:
Dhawal Mehrotra, 25070126060
Bhavya Anup Sharma, 25070126049
'''

In [41]:
!pip install rdkit

In [57]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

In [43]:
df = pd.read_csv("/content/drive/MyDrive/DPEL/MP/Dopamine D2 Receptor.csv", sep = ";")

In [44]:
df.columns

Index(['Molecule ChEMBL ID', 'Molecule Name', 'Molecule Max Phase',
       'Molecular Weight', '#RO5 Violations', 'AlogP', 'Compound Key',
       'Smiles', 'Standard Type', 'Standard Relation', 'Standard Value',
       'Standard Units', 'pChEMBL Value', 'Data Validity Comment', 'Comment',
       'Uo Units', 'Ligand Efficiency BEI', 'Ligand Efficiency LE',
       'Ligand Efficiency LLE', 'Ligand Efficiency SEI', 'Potential Duplicate',
       'Assay ChEMBL ID', 'Assay Description', 'Assay Type', 'BAO Format ID',
       'BAO Label', 'Assay Organism', 'Assay Tissue ChEMBL ID',
       'Assay Tissue Name', 'Assay Cell Type', 'Assay Subcellular Fraction',
       'Assay Parameters', 'Assay Variant Accession', 'Assay Variant Mutation',
       'Target ChEMBL ID', 'Target Name', 'Target Organism', 'Target Type',
       'Document ChEMBL ID', 'Source ID', 'Source Description',
       'Document Journal', 'Document Year', 'Cell ChEMBL ID', 'Properties',
       'Action Type', 'Standard Text Value', 'V

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32479 entries, 0 to 32478
Data columns (total 48 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Molecule ChEMBL ID          32479 non-null  object 
 1   Molecule Name               5111 non-null   object 
 2   Molecule Max Phase          4449 non-null   float64
 3   Molecular Weight            32414 non-null  float64
 4   #RO5 Violations             32067 non-null  float64
 5   AlogP                       32067 non-null  float64
 6   Compound Key                32471 non-null  object 
 7   Smiles                      32355 non-null  object 
 8   Standard Type               32479 non-null  object 
 9   Standard Relation           28939 non-null  object 
 10  Standard Value              23910 non-null  float64
 11  Standard Units              25819 non-null  object 
 12  pChEMBL Value               15052 non-null  float64
 13  Data Validity Comment       189

In [46]:
import pandas as pd

# Creating a copy of the raw dataframe to preserve the original
df_act = df.copy()

# 1. Dropping Administrative & Publication Metadata
# Reason: Journal names, publication years, and source IDs have zero mathematical correlation to molecular physics.
df_act = df_act.drop(columns=[
    'Document ChEMBL ID', 'Source ID', 'Source Description',
    'Document Journal', 'Document Year'
])

# 2. Dropping Constant Target Redundancy
# Reason: We already filtered the database for the Human D2 Receptor. These columns contain the exact same string for all 32,000 rows, offering zero variance for an ML model.
df_act = df_act.drop(columns=[
    'Target ChEMBL ID', 'Target Name', 'Target Organism', 'Target Type'
])

# 3. Dropping Assay Ontologies & Sparse Cellular Data
# Reason: These are highly sparse biological metadata tags (like specific cell lines or mutations) that introduce massive nulls and do not reflect the raw physics of the drug itself.
df_act = df_act.drop(columns=[
    'Assay ChEMBL ID', 'Assay Description', 'BAO Format ID', 'BAO Label',
    'Assay Organism', 'Assay Tissue ChEMBL ID', 'Assay Tissue Name',
    'Assay Cell Type', 'Assay Subcellular Fraction', 'Assay Parameters',
    'Assay Variant Accession', 'Assay Variant Mutation'
])

# 4. Dropping Raw Binding Metrics (Superseded by pChEMBL)
# Reason: 'Standard Value' is measured in varying units and relations. 'pChEMBL Value' is the mathematically standardized, negative logarithmic version of these columns. Retaining both causes extreme multicollinearity.
df_act = df_act.drop(columns=[
    'Standard Type', 'Standard Relation', 'Standard Value',
    'Standard Units', 'Standard Text Value', 'Value', 'Uo Units'
])

# 5. Dropping Alternative Efficiencies
# Reason: To prevent multicollinearity in our Heatmap. We selected 'Ligand Efficiency BEI' (Binding Efficiency Index) for our scope; LE, LLE, and SEI are redundant mathematical derivatives.
df_act = df_act.drop(columns=[
    'Ligand Efficiency LE', 'Ligand Efficiency LLE', 'Ligand Efficiency SEI'
])

# 6. Dropping Sparse Notes & Abandoned Features
# Reason: 'Action Type' due to 93% missing values. The comment columns are unstructured text, unusable by our Random Forest.
df_act = df_act.drop(columns=[
    'Action Type', 'Properties', 'Data Validity Comment', 'Comment'
])

# 7. Dropping Molecule Technicalities and higly missing features
# Reason: 'Molecule Name' is mostly null (many synthetic compounds only have IDs). 'Compound Key' is redundant to 'Molecule ChEMBL ID'.
#Smiles is chemical structure equivalent, which is not required for this project.
df_act = df_act.drop(columns=[
    'Molecule Name', 'Molecule Max Phase', 'Compound Key', 'Potential Duplicate'
])

print(f"Original shape: {df.shape}")
print(f"Purified shape: {df_act.shape}")
print("\nRetained Columns:")
print(df_act.columns.tolist())


Original shape: (32479, 48)
Purified shape: (32479, 9)

Retained Columns:
['Molecule ChEMBL ID', 'Molecular Weight', '#RO5 Violations', 'AlogP', 'Smiles', 'pChEMBL Value', 'Ligand Efficiency BEI', 'Assay Type', 'Cell ChEMBL ID']


In [47]:
df_act.head()

,Molecule ChEMBL ID,Molecular Weight,#RO5 Violations,AlogP,Smiles,pChEMBL Value,Ligand Efficiency BEI,Assay Type,Cell ChEMBL ID
0,CHEMBL7927,326.83,0.0,3.31,Clc1ccc(N2CCN(Cc3cnn4ccccc34)CC2)cc1,5.37,16.42,B,CHEMBL3308072
1,CHEMBL2028019,427.42,0.0,4.34,CN(C)C(=O)N[C@H]1CC[C@H](CCN2CCN(c3cccc(Cl)c3C...,NaN,NaN,B,NaN
2,CHEMBL2028019,427.42,0.0,4.34,CN(C)C(=O)N[C@H]1CC[C@H](CCN2CCN(c3cccc(Cl)c3C...,NaN,NaN,F,NaN
3,CHEMBL2028019,427.42,0.0,4.34,CN(C)C(=O)N[C@H]1CC[C@H](CCN2CCN(c3cccc(Cl)c3C...,NaN,NaN,B,NaN
4,CHEMBL1112,448.39,0.0,4.86,O=C1CCc2ccc(OCCCCN3CCN(c4cccc(Cl)c4Cl)CC3)cc2N1,NaN,NaN,B,NaN


In [48]:
print(df_act.shape)
print(df_act.info())

(32479, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32479 entries, 0 to 32478
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Molecule ChEMBL ID     32479 non-null  object 
 1   Molecular Weight       32414 non-null  float64
 2   #RO5 Violations        32067 non-null  float64
 3   AlogP                  32067 non-null  float64
 4   Smiles                 32355 non-null  object 
 5   pChEMBL Value          15052 non-null  float64
 6   Ligand Efficiency BEI  12782 non-null  float64
 7   Assay Type             32479 non-null  object 
 8   Cell ChEMBL ID         11403 non-null  object 
dtypes: float64(5), object(4)
memory usage: 2.2+ MB
None


In [49]:
# Dropping the highly sparse and physically irrelevant Cell ID
df_act = df_act.drop(columns=['Cell ChEMBL ID'])

In [50]:
df_act.isnull().sum()

,0
Molecule ChEMBL ID,0
Molecular Weight,65
#RO5 Violations,412
AlogP,412
Smiles,124
pChEMBL Value,17427
Ligand Efficiency BEI,19697
Assay Type,0


In [ ]:
# Reason: Instead of statistical guessing (imputation), we
# deterministically calculate the exact Molecular Weight, AlogP (lipophilicity),
# and Lipinski Rule of 5 violations directly from the SMILES atomic structure.
def resurrect_physics(row):
    # Only trigger the heavy RDKit engine if a value is actually missing
    if pd.isna(row['Molecular Weight']) or pd.isna(row['AlogP']) or pd.isna(row['#RO5 Violations']):
        try:
            # Parse the SMILES string into an RDKit molecule object
            mol = Chem.MolFromSmiles(row['Smiles'])

            if mol:
                # Calculate Molecular Weight
                if pd.isna(row['Molecular Weight']):
                    row['Molecular Weight'] = Descriptors.MolWt(mol)

                # Calculate AlogP (Octanol-Water Partition Coefficient)
                if pd.isna(row['AlogP']):
                    row['AlogP'] = Descriptors.MolLogP(mol)

                # Calculate #RO5 Violations (The strict geometric checklist)
                if pd.isna(row['#RO5 Violations']):
                    violations = 0
                    if Descriptors.MolWt(mol) > 500: violations += 1
                    if Descriptors.MolLogP(mol) > 5: violations += 1
                    if Lipinski.NumHDonors(mol) > 5: violations += 1
                    if Lipinski.NumHAcceptors(mol) > 10: violations += 1
                    row['#RO5 Violations'] = violations
        except:
            # Pass silently if a SMILES string is corrupted or unreadable
            pass

    return row

# Applying the resurrection function across the dataframe
df_act = df_act.apply(resurrect_physics, axis=1)

'''
We pull the raw lab measurements back from the
original dataframe. We then perform a highly specific surgical drop:
we only discard a row if it is missing BOTH its standardized pChEMBL
score AND its raw lab measurements.
Reintroduce the raw columns from the original 'df'
(Relying on index alignment to ensure data matches correctly)
'''

df_act['Standard Value'] = df['Standard Value']
df_act['Standard Units'] = df['Standard Units']

# Create a boolean mask for the drop condition
# True if: (Standard Value is NaN OR Standard Units is NaN) AND pChEMBL is NaN
mask = (df_act['Standard Value'].isna() | df_act['Standard Units'].isna()) & df_act['pChEMBL Value'].isna()

# Invert the mask (keep rows that DO NOT match the drop condition)
df_act = df_act[~mask]

# Reset the index to seal the forge
df_act = df_act.reset_index(drop=True)

# Final shape verification
print(f"Reconstruction complete. Current shape: {df_act.shape}")

Reconstruction complete. Current shape: (22959, 10)


In [52]:
print(df_act.columns)
df_act.head()

Index(['Molecule ChEMBL ID', 'Molecular Weight', '#RO5 Violations', 'AlogP',
       'Smiles', 'pChEMBL Value', 'Ligand Efficiency BEI', 'Assay Type',
       'Standard Value', 'Standard Units'],
      dtype='object')


,Molecule ChEMBL ID,Molecular Weight,#RO5 Violations,AlogP,Smiles,pChEMBL Value,Ligand Efficiency BEI,Assay Type,Standard Value,Standard Units
0,CHEMBL7927,326.83,0.0,3.31,Clc1ccc(N2CCN(Cc3cnn4ccccc34)CC2)cc1,5.37,16.42,B,4300.00000,nM
1,CHEMBL1112,448.39,0.0,4.86,O=C1CCc2ccc(OCCCCN3CCN(c4cccc(Cl)c4Cl)CC3)cc2N1,NaN,NaN,B,0.08167,hr
2,CHEMBL1078207,458.59,0.0,4.48,O=S(=O)(Nc1ccc2c(c1)CCN(Cc1cc[nH]n1)CC2)c1ccc(...,6.20,NaN,F,630.96000,nM
3,CHEMBL1094514,418.37,1.0,5.41,Cc1cc(C(O)(c2ccc3c(cnn3-c3ccc(F)cc3)c2)C(F)(F)...,NaN,NaN,B,50.00000,%
4,CHEMBL231546,505.87,2.0,5.60,CN1CCc2cc(Br)c(NS(=O)(=O)c3ccc(-c4ccc(Cl)cc4)c...,7.20,14.23,B,63.10000,nM


In [53]:
df_act.isnull().sum()

,0
Molecule ChEMBL ID,0
Molecular Weight,0
#RO5 Violations,0
AlogP,0
Smiles,0
pChEMBL Value,7932
Ligand Efficiency BEI,10193
Assay Type,0
Standard Value,0
Standard Units,0


In [54]:
unit_counts = df_act['Standard Units'].value_counts(dropna=False)
print("The label of Units:")
print(unit_counts)

The label of Units:
Standard Units
nM                      18901
%                        3830
hr                         66
/min                       63
10'8/M/min                 21
10'7/M/min                 20
mg.kg-1                    16
uM                         14
10'9/M/min                  8
min                         7
M                           4
pmol                        4
10'6/M/min                  4
pM (mg of protein)-1        1
Name: count, dtype: int64


In [55]:
# We retain any row that already has a pChEMBL score
# (regardless of its original unit). For rows missing the score, we ONLY
# keep them if the unit is 'nM', as this is the only dimension we can
# mathematically convert. Everything else (%, hr, /min, etc) is dropped.

survivor_mask = df_act['pChEMBL Value'].notna() | (df_act['Standard Units'] == 'nM')
df_act = df_act[survivor_mask].copy()

# For the rows missing their pChEMBL score but
# possessing a pure nanomolar (nM) measurement, we deterministically
# calculate the score using the negative logarithmic molar formula:
# pChEMBL = 9 - log10(Standard Value).

# Applying the mathematical translation only where pChEMBL is NaN
df_act['pChEMBL Value'] = np.where(
    df_act['pChEMBL Value'].isna(),
    9 - np.log10(df_act['Standard Value']),
    df_act['pChEMBL Value']
)

# Clean up any impossible math anomalies (like log of zero or negatives)
df_act = df_act.replace([np.inf, -np.inf], np.nan)
df_act = df_act.dropna(subset=['pChEMBL Value'])

# 'Ligand Efficiency BEI' is the ratio of binding
# strength to molecular mass. Now that we have a perfectly complete
# pChEMBL column and a fully resurrected Molecular Weight column, we
# can calculate the exact BEI for the 2,270 missing rows.
# Formula: (pChEMBL * 1000) / Molecular Weight

df_act['Ligand Efficiency BEI'] = np.where(
    df_act['Ligand Efficiency BEI'].isna(),
    (df_act['pChEMBL Value'] * 1000) / df_act['Molecular Weight'],
    df_act['Ligand Efficiency BEI']
)
df_act = df_act.drop(columns=['Standard Value', 'Standard Units']) #now they are redundant, once again.

# Seal the forge and reset the ledger
df_act = df_act.reset_index(drop=True)

# Final verification of the architecture
print(f"Phase 1 Complete. The dataset is pure. Final rows: {df_act.shape}")
print(f"Missing values remaining in df_act:\n{df_act.isna().sum()}")

Phase 1 Complete. The dataset is pure. Final rows: (18896, 8)
Missing values remaining in df_act:
Molecule ChEMBL ID       0
Molecular Weight         0
#RO5 Violations          0
AlogP                    0
Smiles                   0
pChEMBL Value            0
Ligand Efficiency BEI    0
Assay Type               0
dtype: int64


/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/usr/local/lib/python3.12/dist-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [60]:
df_act['Activity_Class'] = (df_act['pChEMBL Value'] >= 6.0).astype(int)
# Reason: We convert the continuous biological spectrum into a deterministic threshold.
# A pChEMBL score of 6.0 (which equals 1 microMolar)
# is the industry standard boundary for a "successful" connection.
encoder = LabelEncoder()
df_act['Assay Type'] = encoder.fit_transform(df_act['Assay Type'])
# Reason: Machine learning models require numerical syntax.
# We translate the categorical 'Assay Type' (B, F, A, etc.) into a
# machine-readable integer matrix using Scikit-Learn's LabelEncoder.

scaler = MinMaxScaler()
physics_features = ['Molecular Weight', 'AlogP', 'Ligand Efficiency BEI']
for feature in physics_features:
    df_act[f'std_{feature}'] = scaler.fit_transform(df_act[[feature]])
# While Standard Scaler relies heavily on standard
# deviations and the assumption of a normal distribution; the messy,
# abstract statistics we need to avoid for varying dataset like this, MinMaxScaler is pure geometry.
# It mathematically crushes the physical dimensions into a strict,
# bounded box between 0 and 1. Since our Random Forest algorithm draws
# literal boxes (decision boundaries) to split data, feeding it a
# strictly bounded reality allows it to optimize its splits with absolute
# certainty, preventing massive features like 'Molecular Weight' from
# overshadowing the delicate decimal scales of 'AlogP'.


df_act.head()

,Molecule ChEMBL ID,Molecular Weight,#RO5 Violations,AlogP,Smiles,pChEMBL Value,Ligand Efficiency BEI,Assay Type,Activity_Class,std_Molecular Weight,std_AlogP,std_Ligand Efficiency BEI
0,CHEMBL7927,326.83,0.0,3.31,Clc1ccc(N2CCN(Cc3cnn4ccccc34)CC2)cc1,5.37,16.420000,1,0,0.066335,0.487544,0.178423
1,CHEMBL1078207,458.59,0.0,4.48,O=S(=O)(Nc1ccc2c(c1)CCN(Cc1cc[nH]n1)CC2)c1ccc(...,6.20,13.519702,2,1,0.102051,0.528780,0.146908
2,CHEMBL231546,505.87,2.0,5.60,CN1CCc2cc(Br)c(NS(=O)(=O)c3ccc(-c4ccc(Cl)cc4)c...,7.20,14.230000,1,1,0.114867,0.568254,0.154626
3,CHEMBL231431,444.96,0.0,4.98,CN1CCc2ccc(NS(=O)(=O)c3ccc(-c4ccc(F)c(Cl)c4)cc...,7.30,16.410000,1,1,0.098356,0.546402,0.178315
4,CHEMBL231333,358.51,0.0,3.82,CCCCc1ccc(S(=O)(=O)Nc2ccc3c(c2)CN(C)CC3)cc1,6.40,17.850000,1,1,0.074922,0.505519,0.193962


In [61]:
save_path = "/content/drive/MyDrive/DPEL/MP/Processed DataFrame.csv"

# Writing the mathematical marble to the disk
df_act.to_csv(save_path, sep=';', index=False)